# ProtocolX — Phase I video (voices + assembly) · Kaggle version

**Before running** (right-hand *Settings* panel of this notebook):
- **Accelerator → GPU T4 x2** (not P100: current PyTorch no longer supports it).
- **Internet → On** (needs a phone-verified Kaggle account).
- **Add Input → Datasets → Your Datasets → `protocolx-bundle`** (the private dataset you made from `colab_bundle.zip`).

Then **Run All**. The video and voices appear under **Output** (right panel) and as download links at the end.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install f5-tts faster-whisper num2words
!apt-get -qq install -y fonts-dejavu-core ffmpeg > /dev/null 2>&1 || true
!ffmpeg -version | head -1

In [ ]:
# Find the uploaded bundle under /kaggle/input and copy it to a writable folder
import os, shutil, glob, zipfile
hits = glob.glob('/kaggle/input/**/script.json', recursive=True)
if not hits:  # dataset kept as a zip instead of being extracted
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, 'Bundle not found: add the protocolx-bundle dataset via Add Input.'
    zipfile.ZipFile(zips[0]).extractall('/kaggle/working/work')
else:
    shutil.copytree(os.path.dirname(hits[0]), '/kaggle/working/work', dirs_exist_ok=True)
os.chdir('/kaggle/working/work')
print(sorted(os.listdir('.')))
print(len(os.listdir('visuals/out')), 'stills;', sorted(os.listdir('ref')))

In [ ]:
# Settings
SPEED = 1.0          # raise to 1.05–1.10 if the final video is longer than 5:00
MAX_TRIES = 3        # regenerate a line up to this many times if Whisper says it doesn't match
MIN_MATCH = 0.80     # 0..1 similarity between intended text and what Whisper hears

In [ ]:
# Generate every voice line, verify it, retry if needed
import os, json, re, difflib, subprocess
import numpy as np
from f5_tts.api import F5TTS
from faster_whisper import WhisperModel
from num2words import num2words

# F5-TTS clips references to 12 s; keep Shagun's ref under that with a transcript that matches exactly.
if float(subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','csv=p=0','ref/shagun.wav'],
                        capture_output=True, text=True).stdout) > 11.5:
    subprocess.run(['ffmpeg','-v','error','-y','-i','ref/shagun.wav','-t','7.6','/tmp/s.wav'], check=True)
    os.replace('/tmp/s.wav', 'ref/shagun.wav')
    open('ref/shagun.txt','w').write('Hello everyone, we are presenting our project on credit risk scoring and loan default prediction.')

script = json.load(open('script.json'))
tts = F5TTS()

def load16k(path):
    # Decode with ffmpeg (not PyAV, whose installed version breaks faster-whisper's file loader).
    raw = subprocess.run(['ffmpeg','-v','error','-i',path,'-f','s16le','-ac','1','-ar','16000','-'],
                         capture_output=True, check=True).stdout
    return np.frombuffer(raw, np.int16).astype(np.float32) / 32768

try:
    asr = WhisperModel('small.en', device='cuda', compute_type='float16')
    list(asr.transcribe(load16k('ref/aditya.wav'))[0])  # fail fast if cuDNN is missing
except Exception as e:
    print('Whisper on GPU failed, using CPU:', e)
    asr = WhisperModel('small.en', device='cpu', compute_type='int8')
os.makedirs('voices', exist_ok=True)

def letters(s):
    s = re.sub(r'\d+(\.\d+)?', lambda m: ' ' + num2words(float(m.group()) if '.' in m.group() else int(m.group())) + ' ', s)
    return re.sub(r'[^a-z]', '', s.lower())

def heard(path):
    segs, _ = asr.transcribe(load16k(path), beam_size=5)
    return ' '.join(s.text.strip() for s in segs)

log = []
for m in script['members']:
    ref = f"ref/{m['name']}.wav"
    ref_text = open(f"ref/{m['name']}.txt").read().strip() + ' '
    for i, line in enumerate(m['lines'], 1):
        out = f"voices/{m['name']}_{i:02d}.wav"
        best = (-1, None)
        for t in range(MAX_TRIES):
            tmp = f"/tmp/try_{t}.wav"
            tts.infer(ref_file=ref, ref_text=ref_text, gen_text=line['tts'], file_wave=tmp,
                      seed=1000 + 17 * t + i, speed=SPEED, remove_silence=False, show_info=lambda *a, **k: None)
            h = heard(tmp)
            score = difflib.SequenceMatcher(None, letters(line['tts']), letters(h)).ratio()
            if score > best[0]:
                best = (score, h)
                os.replace(tmp, out)
            if score >= MIN_MATCH:
                break
        log.append((m['name'], i, round(best[0], 2), best[1]))
        flag = '' if best[0] >= MIN_MATCH else '   <-- CHECK THIS LINE'
        print(f"{m['name']:7} {i:02d}  match={best[0]:.2f}{flag}\n        heard: {best[1]}")
bad = [l for l in log if l[2] < MIN_MATCH]
print('\nAll lines matched.' if not bad else f'\n{len(bad)} line(s) below {MIN_MATCH}: listen to them below.')


In [ ]:
# Listen: first line of each member (and any flagged lines)
from IPython.display import Audio, display
for m in script['members']:
    print(m['name']); display(Audio(f"voices/{m['name']}_01.wav"))
for name, i, score, h in log:
    if score < MIN_MATCH:
        print('FLAGGED', name, i, score); display(Audio(f'voices/{name}_{i:02d}.wav'))

In [ ]:
# Assemble the video (timing comes from the audio)
!python assemble.py --script script.json --voices voices --stills visuals/out --out protocolx_phase1.mp4

In [ ]:
# Preview inside Colab
from IPython.display import Video
Video('protocolx_phase1.mp4', embed=True, width=960)

In [ ]:
# Download links (files are also listed under Output in the right panel)
import shutil
from IPython.display import FileLink, display
shutil.make_archive('/kaggle/working/voices', 'zip', 'voices')
shutil.copy('protocolx_phase1.mp4', '/kaggle/working/protocolx_phase1.mp4')
os.chdir('/kaggle/working')
display(FileLink('protocolx_phase1.mp4'), FileLink('voices.zip'))